In [ ]:
!pip install catboost shap

In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import shap
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from catboost import CatBoostRegressor, Pool

DATA_PATH = 'nigeria_agri_yield_enhanced.csv'

def engineer_features(df):
    print("\nEngineering features...")
    df['soil_fertility_index'] = df['soil_nitrogen'] + df['soil_phosphorus'] + df['soil_potassium']
    df['np_ratio'] = df['soil_nitrogen'] / (df['soil_phosphorus'] + 1e-6)
    df['climate_index'] = df['rainfall_mm'] * df['temperature_C']
    return df

def prepare_data(df):
    target_col = 'yield_kg_ha'
    
    # Drop non-causal and unused tags
    df = df.drop(columns=['region', 'state', 'pest_type', 'pest_severity', 'labor_input', 'farm_size_ha', 'rainfall_variability'], errors='ignore')
    
    # Drop rows with NaN values to prevent CatBoost from crashing on categorical NaNs
    print(f"Rows before dropping NaNs: {len(df)}")
    df = df.dropna()
    print(f"Rows after dropping NaNs: {len(df)}")
    
    X = df.drop(columns=[target_col], errors='ignore')
    y = df[target_col]
    
    categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
    numeric_cols = X.select_dtypes(include=['number']).columns.tolist()
    
    # Ensure all categorical columns are strings and handle any edge case NaNs
    for col in categorical_cols:
        X[col] = X[col].astype(str)
        
    return X, y, categorical_cols, numeric_cols

def train():
    if not os.path.exists(DATA_PATH):
        print(f"Error: {DATA_PATH} not found. Please upload the dataset to Colab.")
        return
        
    df = pd.read_csv(DATA_PATH)
    df = engineer_features(df)
    X, y, cat_features, num_features = prepare_data(df)

    # Train Final Model
    model = CatBoostRegressor(
        iterations=400,
        learning_rate=0.08,
        depth=6,
        loss_function='RMSE',
        eval_metric='RMSE',
        random_seed=42,
        cat_features=cat_features,
        verbose=100
    )

    model.fit(X, y)
    model.save_model('catboost_model.cbm')
    print("Model saved to catboost_model.cbm")

train()